# Conser-vision Wildlife Classification

In [1]:
import importlib.util, subprocess, sys

def ensure(name, pip_name=None):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or name])

ensure('timm', 'timm>=1.0.19')
ensure('albumentations', 'albumentations>=1.4.20')

import os, gc, random, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
from timm.utils import ModelEmaV2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import log_loss, accuracy_score
from scipy.optimize import minimize_scalar
warnings.filterwarnings('ignore')

SEED=2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
# if not torch.cuda.is_available(): raise RuntimeError('Enable Kaggle GPU first.')
# print('GPU:', torch.cuda.get_device_name(0))

Device: cuda


# Configuration 

In [2]:
RUN_MODE='full'       # change to 'full' after the smoke run succeeds
MODEL_NAME='convnext_base.fb_in22k_ft_in1k_384'
IMAGE_SIZE=384
N_SPLITS=5
BATCH_SIZE=8
NUM_WORKERS=0
GRAD_ACCUM=2
BACKBONE_LR=2e-5
HEAD_LR=2e-4
WEIGHT_DECAY=1e-4
MIXUP_ALPHA=0.20
MIXUP_PROB=0.50
LABEL_SMOOTHING=0.05
PATIENCE=4

#Use smoke to test it out
if RUN_MODE=='smoke':
    EPOCHS=2; FOLDS_TO_RUN=[0]
else:
    EPOCHS=14; FOLDS_TO_RUN=list(range(N_SPLITS))

print('Mode:', RUN_MODE, '| folds:', FOLDS_TO_RUN, '| epochs:', EPOCHS)

Mode: full | folds: [0, 1, 2, 3, 4] | epochs: 14


# Locate competition files

In [3]:
INPUT_ROOT=Path('/kaggle/input')
for p in sorted(INPUT_ROOT.iterdir()): print('-',p.name)

def hit(name):
    xs=list(INPUT_ROOT.rglob(name))
    return xs[0] if xs else None

P={k:hit(v) for k,v in {
 'train_features':'train_features.csv','train_labels':'train_labels.csv',
 'test_features':'test_features.csv','submission_format':'submission_format.csv'}.items()}
if not all(P.values()): raise FileNotFoundError(P)
for k,v in P.items(): print(k, '->', v)

train_features=pd.read_csv(P['train_features'])
train_labels=pd.read_csv(P['train_labels'])
test_features=pd.read_csv(P['test_features'])
submission_format=pd.read_csv(P['submission_format'])
class_names=[c for c in train_labels.columns if c!='id']
NUM_CLASSES=len(class_names)
train_df=train_features.merge(train_labels,on='id',validate='one_to_one')
train_df['target']=train_df[class_names].to_numpy().argmax(1)

def infer_root(df):
    rel=Path(df.iloc[0].filepath)
    for base in [P['train_features'].parent,P['train_features'].parent.parent,INPUT_ROOT]:
        if (base/rel).exists(): return base
    for full in INPUT_ROOT.rglob(rel.name):
        if tuple(full.parts[-len(rel.parts):])==tuple(rel.parts):
            return Path(*full.parts[:-len(rel.parts)])
    raise FileNotFoundError(rel)
IMAGE_ROOT=infer_root(train_df)
print('Train:',len(train_df),'Test:',len(test_features),'Classes:',class_names)
print('Train sites:',train_df.site.nunique(),'Test sites:',test_features.site.nunique())
print('Site overlap:',len(set(train_df.site)&set(test_features.site)))
print('Image root:',IMAGE_ROOT)


- datasets
train_features -> /kaggle/input/datasets/bipin886/competition-vfipjyh/train_features.csv
train_labels -> /kaggle/input/datasets/bipin886/competition-vfipjyh/train_labels.csv
test_features -> /kaggle/input/datasets/bipin886/competition-vfipjyh/test_features.csv
submission_format -> /kaggle/input/datasets/bipin886/competition-vfipjyh/submission_format.csv
Train: 16488 Test: 4464 Classes: ['antelope_duiker', 'bird', 'blank', 'civet_genet', 'hog', 'leopard', 'monkey_prosimian', 'rodent']
Train sites: 148 Test sites: 51
Site overlap: 0
Image root: /kaggle/input/datasets/bipin886/competition-vfipjyh


# Site-disjoint folds and basic audit

In [4]:
print(train_df[class_names].sum().sort_values(ascending=False).astype(int))
cv=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
train_df['fold']=-1
for f,(_,vi) in enumerate(cv.split(train_df,train_df.target,groups=train_df.site)):
    train_df.loc[vi,'fold']=f
train_df['fold']=train_df.fold.astype(int)
for f in range(N_SPLITS):
    assert set(train_df.loc[train_df.fold!=f,'site']).isdisjoint(set(train_df.loc[train_df.fold==f,'site']))
print(pd.crosstab(train_df.fold,train_df.target))
train_df[['id','site','fold']].to_csv('/kaggle/working/site_folds.csv',index=False)

monkey_prosimian    2492
antelope_duiker     2474
civet_genet         2423
leopard             2254
blank               2213
rodent              2013
bird                1641
hog                  978
dtype: int64
target    0    1    2     3    4    5    6    7
fold                                           
0       322  196  490   115  154  290  247  363
1       297  222  377   326  166  170  286   68
2       524  393  484   421  263  601  669  792
3       866  288  422   384  145  904  565  359
4       465  542  440  1177  250  289  725  431


# Dataset and transforms

In [5]:
train_tfms=A.Compose([
    A.RandomResizedCrop(size=(IMAGE_SIZE,IMAGE_SIZE),scale=(0.72,1.0),ratio=(0.85,1.18),p=1),
    A.HorizontalFlip(p=.5),
    A.ColorJitter(brightness=.25,contrast=.25,saturation=.15,hue=.02,p=.5),
    A.ToGray(p=.08), A.GaussianBlur(blur_limit=(3,5),p=.08),
    A.Normalize(mean=(.485,.456,.406),std=(.229,.224,.225)), ToTensorV2()
])
valid_tfms=A.Compose([
    A.Resize(IMAGE_SIZE,IMAGE_SIZE),
    A.Normalize(mean=(.485,.456,.406),std=(.229,.224,.225)), ToTensorV2()
])

class WildlifeDataset(Dataset):
    def __init__(self,df,training=True):
        self.df=df.reset_index(drop=True); self.training=training
        self.tfms=train_tfms if training else valid_tfms
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        im=np.array(Image.open(IMAGE_ROOT/r.filepath).convert('RGB'))
        im=self.tfms(image=im)['image']
        return (im,int(r.target)) if self.training else (im,str(r.id))

# Model and helpers

In [6]:
def build_model(pretrained=True):
    return timm.create_model(
        MODEL_NAME,
        pretrained=pretrained,
        num_classes=NUM_CLASSES,
        drop_rate=.15
    )
    
def build_optimizer(model):
    head=model.get_classifier(); ids={id(p) for p in head.parameters()}
    bb=[p for p in model.parameters() if id(p) not in ids and p.requires_grad]
    hp=[p for p in model.parameters() if id(p) in ids and p.requires_grad]
    return torch.optim.AdamW([{'params':bb,'lr':BACKBONE_LR},{'params':hp,'lr':HEAD_LR}],weight_decay=WEIGHT_DECAY)

def softmax_np(z,T=1.):
    z=z/T; z-=z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)

def score(y,z,T=1.):
    p=softmax_np(z,T); return log_loss(y,p,labels=np.arange(NUM_CLASSES)), accuracy_score(y,p.argmax(1))

mixup=Mixup(mixup_alpha=MIXUP_ALPHA,cutmix_alpha=0.,prob=MIXUP_PROB,switch_prob=0.,mode='batch',label_smoothing=LABEL_SMOOTHING,num_classes=NUM_CLASSES)
criterion=SoftTargetCrossEntropy()

# ======================================================
# TARGET LOOKUP FOR VALIDATION
# ======================================================

TARGET_BY_ID = dict(
    zip(
        train_df["id"].astype(str),
        train_df["target"].astype(int)
    )
)


# ======================================================
# INFERENCE / VALIDATION FUNCTION
# ======================================================

@torch.no_grad()
def infer(
    model,
    loader,
    tta=False,
    test=False
):

    model.eval()

    zs = []
    info = []

    # --------------------------------------------------
    # PROGRESS BAR NAME
    # --------------------------------------------------

    if test:

        description = "Test prediction"

    elif tta:

        description = "Validation TTA"

    else:

        description = "Validation"


    progress = tqdm(
        loader,
        desc=description,
        leave=False
    )


    # --------------------------------------------------
    # INFERENCE LOOP
    # --------------------------------------------------

    for x, y in progress:

        x = x.to(
            DEVICE,
            non_blocking=True
        )


        # --------------------------------------------------
        # FORWARD PASS
        # --------------------------------------------------

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16
        ):

            z = model(
                x
            )


            # ----------------------------------------------
            # TEST-TIME AUGMENTATION
            # ----------------------------------------------

            if tta:

                z_flip = model(
                    torch.flip(
                        x,
                        dims=[3]
                    )
                )

                z = (
                    z
                    +
                    z_flip
                ) / 2.0


        # --------------------------------------------------
        # SAVE LOGITS
        # --------------------------------------------------

        zs.append(
            z.float()
            .cpu()
            .numpy()
        )


        # --------------------------------------------------
        # TEST DATA
        # --------------------------------------------------

        if test:

            # y contains image IDs
            info.extend(
                [
                    str(item)
                    for item in y
                ]
            )


        # --------------------------------------------------
        # VALIDATION DATA
        # --------------------------------------------------

        else:

            # Case 1:
            # Dataset directly returned numerical targets.
            if torch.is_tensor(y):

                info.append(
                    y.cpu()
                    .numpy()
                    .astype(
                        np.int64
                    )
                )


            # Case 2:
            # Current WildlifeDataset with training=False
            # returns image IDs.
            # Convert IDs back to their true labels.
            else:

                batch_ids = [
                    str(item)
                    for item in y
                ]

                batch_targets = np.array(
                    [
                        TARGET_BY_ID[
                            image_id
                        ]
                        for image_id
                        in batch_ids
                    ],
                    dtype=np.int64
                )

                info.append(
                    batch_targets
                )


    # --------------------------------------------------
    # CONCATENATE LOGITS
    # --------------------------------------------------

    zs = np.concatenate(
        zs,
        axis=0
    )


    # --------------------------------------------------
    # FORMAT OUTPUT
    # --------------------------------------------------

    if test:

        info_out = info

    else:

        info_out = np.concatenate(
            info,
            axis=0
        )


    return (
        zs,
        info_out
    )

# Train fold(s), create OOF and test logits

In [7]:
oof = np.full(
    (len(train_df), NUM_CLASSES),
    np.nan,
    dtype=np.float32
)

y_all = train_df.target.to_numpy()

test_fold_logits = []
fold_rows = []
test_ids = None


for fold in FOLDS_TO_RUN:

    print("\n" + "=" * 70)
    print(f"FOLD {fold}")
    print("=" * 70)

    # --------------------------------------------------
    # SPLIT DATA
    # --------------------------------------------------

    tr = train_df[
        train_df.fold != fold
    ].copy()

    va = train_df[
        train_df.fold == fold
    ].copy()

    # --------------------------------------------------
    # DATALOADERS
    # --------------------------------------------------

    tr_loader = DataLoader(
        WildlifeDataset(tr, True),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=True
    )

    va_loader = DataLoader(
        WildlifeDataset(va, False),
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    te_loader = DataLoader(
        WildlifeDataset(test_features, False),
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    # --------------------------------------------------
    # CHECKPOINT PATHS
    # --------------------------------------------------

    run_tag = str(RUN_MODE).lower()

    safe_model_name = MODEL_NAME.replace(
        "/",
        "_"
    )

    best_ck = (
        f"/kaggle/working/"
        f"{safe_model_name}"
        f"_fold{fold}_{run_tag}_best.pt"
    )

    resume_ck = (
        f"/kaggle/working/"
        f"{safe_model_name}"
        f"_fold{fold}_{run_tag}_resume.pt"
    )

    resume_exists = os.path.exists(
        resume_ck
    )

    print(
        f"Best checkpoint:   {best_ck}"
    )

    print(
        f"Resume checkpoint: {resume_ck}"
    )

    if resume_exists:

        print(
            "✓ Resume checkpoint found."
        )

    # --------------------------------------------------
    # BUILD TRAINING MODEL
    # --------------------------------------------------

    # If a resume checkpoint exists, pretrained=False
    # prevents another Hugging Face download.
    # The checkpoint itself contains the trained weights.

    model = build_model(
        pretrained=not resume_exists
    ).to(
        DEVICE
    )

    # --------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------

    opt = build_optimizer(
        model
    )

    # --------------------------------------------------
    # SCHEDULER
    # --------------------------------------------------

    sch = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=max(
            EPOCHS,
            1
        ),
        eta_min=BACKBONE_LR * 0.05
    )

    # --------------------------------------------------
    # MIXED PRECISION
    # --------------------------------------------------

    scaler = torch.amp.GradScaler(
        "cuda"
    )

    # --------------------------------------------------
    # EMA
    # --------------------------------------------------

    ema = ModelEmaV2(
        model,
        decay=0.9997
    )

    # --------------------------------------------------
    # DEFAULT TRAINING STATE
    # --------------------------------------------------

    best = float("inf")

    best_epoch = -1

    stale = 0

    start_epoch = 1

    # --------------------------------------------------
    # RESTORE RESUME CHECKPOINT
    # --------------------------------------------------

    if resume_exists:

        checkpoint = torch.load(
            resume_ck,
            map_location=DEVICE,
            weights_only=False
        )

        # ----------------------------------------------
        # SAFETY CHECKS
        # ----------------------------------------------

        if checkpoint["fold"] != fold:

            raise ValueError(
                "Checkpoint fold does not match "
                "the current fold."
            )

        if checkpoint["model_name"] != MODEL_NAME:

            raise ValueError(
                "Checkpoint model does not match "
                "MODEL_NAME."
            )

        if checkpoint["num_classes"] != NUM_CLASSES:

            raise ValueError(
                "Checkpoint NUM_CLASSES does "
                "not match."
            )

        if (
            checkpoint.get(
                "run_mode",
                RUN_MODE
            )
            != RUN_MODE
        ):

            raise ValueError(
                "Checkpoint RUN_MODE does not "
                "match current RUN_MODE."
            )

        if (
            checkpoint.get(
                "planned_epochs",
                EPOCHS
            )
            != EPOCHS
        ):

            raise ValueError(
                f"Checkpoint was created with "
                f"EPOCHS="
                f"{checkpoint.get('planned_epochs')}, "
                f"but current EPOCHS={EPOCHS}."
            )

        # ----------------------------------------------
        # RESTORE MODEL
        # ----------------------------------------------

        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        # ----------------------------------------------
        # RESTORE EMA
        # ----------------------------------------------

        ema.module.load_state_dict(
            checkpoint[
                "ema_state_dict"
            ]
        )

        # ----------------------------------------------
        # RESTORE OPTIMIZER
        # ----------------------------------------------

        opt.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )

        # ----------------------------------------------
        # RESTORE SCHEDULER
        # ----------------------------------------------

        sch.load_state_dict(
            checkpoint[
                "scheduler_state_dict"
            ]
        )

        # ----------------------------------------------
        # RESTORE SCALER
        # ----------------------------------------------

        scaler.load_state_dict(
            checkpoint[
                "scaler_state_dict"
            ]
        )

        # ----------------------------------------------
        # RESTORE TRAINING STATE
        # ----------------------------------------------

        best = checkpoint[
            "best"
        ]

        best_epoch = checkpoint[
            "best_epoch"
        ]

        stale = checkpoint[
            "stale"
        ]

        start_epoch = (
            checkpoint[
                "epoch"
            ]
            + 1
        )

        # ----------------------------------------------
        # RESTORE RANDOM STATES
        # ----------------------------------------------

        if "python_rng_state" in checkpoint:

            random.setstate(
                checkpoint[
                    "python_rng_state"
                ]
            )

        if "numpy_rng_state" in checkpoint:

            np.random.set_state(
                checkpoint[
                    "numpy_rng_state"
                ]
            )

        if "torch_rng_state" in checkpoint:

            torch.set_rng_state(
                checkpoint[
                    "torch_rng_state"
                ].cpu()
            )

        cuda_rng_state = checkpoint.get(
            "cuda_rng_state"
        )

        if (
            torch.cuda.is_available()
            and
            cuda_rng_state is not None
        ):

            torch.cuda.set_rng_state_all(
                [
                    state.cpu()
                    for state
                    in cuda_rng_state
                ]
            )

        print(
            f"✓ Resuming Fold {fold} | "
            f"last completed epoch: "
            f"{checkpoint['epoch']} | "
            f"next epoch: "
            f"{start_epoch} | "
            f"best LogLoss: "
            f"{best:.5f}"
        )

        # If this fold had already early-stopped
        # before the interruption, do not train
        # another unnecessary epoch.

        if (
            RUN_MODE == "full"
            and
            stale >= PATIENCE
        ):

            print(
                f"✓ Fold {fold} had already "
                f"reached early stopping."
            )

            start_epoch = (
                EPOCHS + 1
            )

    else:

        print(
            f"No resume checkpoint. "
            f"Starting Fold {fold} "
            f"from epoch 1."
        )

    # ==================================================
    # TRAINING
    # ==================================================

    for ep in range(
        start_epoch,
        EPOCHS + 1
    ):

        model.train()

        run = 0.0

        opt.zero_grad(
            set_to_none=True
        )

        t0 = time.time()

        # ----------------------------------------------
        # TQDM TRAINING BAR
        # ----------------------------------------------

        progress = tqdm(
            tr_loader,
            desc=(
                f"Fold {fold} | "
                f"Epoch {ep}/{EPOCHS}"
            ),
            leave=False
        )

        # ----------------------------------------------
        # TRAINING BATCHES
        # ----------------------------------------------

        for step, (x, y) in enumerate(
            progress,
            1
        ):

            x = x.to(
                DEVICE,
                non_blocking=True
            )

            y = y.to(
                DEVICE,
                non_blocking=True
            )

            # ------------------------------------------
            # MIXUP
            # ------------------------------------------

            x, ys = mixup(
                x,
                y
            )

            # ------------------------------------------
            # FORWARD PASS
            # ------------------------------------------

            with torch.amp.autocast(
                "cuda",
                dtype=torch.float16
            ):

                logits = model(
                    x
                )

                loss = (
                    criterion(
                        logits,
                        ys
                    )
                    /
                    GRAD_ACCUM
                )

            # ------------------------------------------
            # BACKPROP
            # ------------------------------------------

            scaler.scale(
                loss
            ).backward()

            run += (
                loss.item()
                *
                GRAD_ACCUM
            )

            # ------------------------------------------
            # OPTIMIZER STEP
            # ------------------------------------------

            if (
                step % GRAD_ACCUM == 0
                or
                step == len(tr_loader)
            ):

                scaler.unscale_(
                    opt
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    2.0
                )

                scaler.step(
                    opt
                )

                scaler.update()

                opt.zero_grad(
                    set_to_none=True
                )

                ema.update(
                    model
                )

            # ------------------------------------------
            # LIVE PROGRESS INFORMATION
            # ------------------------------------------

            progress.set_postfix(
                loss=(
                    f"{loss.item() * GRAD_ACCUM:.4f}"
                ),
                lr=(
                    f"{opt.param_groups[0]['lr']:.2e}"
                )
            )

        # ==================================================
        # VALIDATION
        # ==================================================

        val_logits, val_targets = infer(
            ema.module,
            va_loader,
            tta=False,
            test=False
        )

        val_probs = softmax_np(
            val_logits
        )

        val_loss = log_loss(
            val_targets,
            val_probs,
            labels=np.arange(
                NUM_CLASSES
            )
        )

        val_acc = (
            val_probs.argmax(
                1
            )
            ==
            val_targets
        ).mean()

        # --------------------------------------------------
        # SCHEDULER STEP
        # --------------------------------------------------

        sch.step()

        # --------------------------------------------------
        # EPOCH STATISTICS
        # --------------------------------------------------

        epoch_time = (
            time.time()
            -
            t0
        ) / 60

        train_loss = (
            run
            /
            max(
                len(tr_loader),
                1
            )
        )

        # --------------------------------------------------
        # UPDATE BEST MODEL
        # --------------------------------------------------

        if val_loss < best:

            best = val_loss

            best_epoch = ep

            stale = 0

            torch.save(
                ema.module.state_dict(),
                best_ck
            )

            print(
                f"✓ New best model saved: "
                f"{best:.5f}"
            )

        else:

            stale += 1

        # --------------------------------------------------
        # SAVE FULL RESUME CHECKPOINT
        # --------------------------------------------------

        checkpoint = {

            "fold":
                fold,

            "epoch":
                ep,

            "model_state_dict":
                model.state_dict(),

            "ema_state_dict":
                ema.module.state_dict(),

            "optimizer_state_dict":
                opt.state_dict(),

            "scheduler_state_dict":
                sch.state_dict(),

            "scaler_state_dict":
                scaler.state_dict(),

            "best":
                best,

            "best_epoch":
                best_epoch,

            "stale":
                stale,

            "model_name":
                MODEL_NAME,

            "num_classes":
                NUM_CLASSES,

            "run_mode":
                RUN_MODE,

            "planned_epochs":
                EPOCHS,

            "python_rng_state":
                random.getstate(),

            "numpy_rng_state":
                np.random.get_state(),

            "torch_rng_state":
                torch.get_rng_state(),

            "cuda_rng_state": (
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available()
                else None
            )
        }

        torch.save(
            checkpoint,
            resume_ck
        )

        # --------------------------------------------------
        # DISPLAY EPOCH RESULT
        # --------------------------------------------------

        print(
            f"Fold {fold} | "
            f"Epoch {ep}/{EPOCHS} | "
            f"Train Loss: "
            f"{train_loss:.4f} | "
            f"Val LogLoss: "
            f"{val_loss:.5f} | "
            f"Val Acc: "
            f"{val_acc:.4f} | "
            f"Best: "
            f"{best:.5f} | "
            f"Best Epoch: "
            f"{best_epoch} | "
            f"Stale: "
            f"{stale}/{PATIENCE} | "
            f"Time: "
            f"{epoch_time:.1f} min"
        )

        print(
            f"✓ Resume checkpoint saved: "
            f"Epoch {ep}"
        )

        # --------------------------------------------------
        # EARLY STOPPING
        # --------------------------------------------------

        if (
            RUN_MODE == "full"
            and
            stale >= PATIENCE
        ):

            print(
                f"Early stopping at "
                f"epoch {ep}"
            )

            break

    # ==================================================
    # LOAD BEST MODEL AFTER TRAINING
    # ==================================================

    print(
        f"\nLoading best Fold "
        f"{fold} model..."
    )

    if not os.path.exists(
        best_ck
    ):

        raise FileNotFoundError(
            f"Best checkpoint not found: "
            f"{best_ck}"
        )

    best_model = build_model(
        pretrained=False
    ).to(
        DEVICE
    )

    best_model.load_state_dict(
        torch.load(
            best_ck,
            map_location=DEVICE
        )
    )

    best_model.eval()

    print(
        f"✓ Best Fold {fold} "
        f"model loaded."
    )

    # ==================================================
    # TTA VALIDATION
    # ==================================================

    val_logits, val_targets = infer(
        best_model,
        va_loader,
        tta=True,
        test=False
    )

    val_probs = softmax_np(
        val_logits
    )

    tta_loss = log_loss(
        val_targets,
        val_probs,
        labels=np.arange(
            NUM_CLASSES
        )
    )

    tta_acc = (
        val_probs.argmax(
            1
        )
        ==
        val_targets
    ).mean()

    print(
        f"Fold {fold} "
        f"TTA LogLoss: "
        f"{tta_loss:.5f}"
    )

    print(
        f"Fold {fold} "
        f"TTA Accuracy: "
        f"{tta_acc:.4f}"
    )

    # ==================================================
    # SAVE OOF PREDICTIONS
    # ==================================================

    oof[
        va.index.to_numpy()
    ] = val_logits

    # ==================================================
    # TEST PREDICTIONS
    # ==================================================

    te_logits, ids = infer(
        best_model,
        te_loader,
        tta=True,
        test=True
    )

    test_fold_logits.append(
        te_logits.astype(
            np.float32
        )
    )

    if test_ids is None:

        test_ids = list(
            ids
        )

    else:

        if list(ids) != list(
            test_ids
        ):

            raise RuntimeError(
                "Test image order changed "
                "between folds."
            )

    # ==================================================
    # SAVE FOLD RESULTS
    # ==================================================

    fold_rows.append(
        {
            "fold":
                fold,

            "best_epoch":
                best_epoch,

            "best_val_logloss":
                best,

            "tta_logloss":
                tta_loss,

            "tta_accuracy":
                tta_acc
        }
    )

    # ==================================================
    # SAVE CURRENT PREDICTION STATE
    # ==================================================

    np.save(
        "/kaggle/working/"
        "oof_logits.npy",
        oof
    )

    np.save(
        "/kaggle/working/"
        "y_all.npy",
        y_all
    )

    np.save(
        "/kaggle/working/"
        "test_fold_logits.npy",
        np.stack(
            test_fold_logits,
            axis=0
        )
    )

    pd.DataFrame(
        fold_rows
    ).to_csv(
        "/kaggle/working/"
        "fold_results.csv",
        index=False
    )

    # ==================================================
    # CLEAN GPU MEMORY
    # ==================================================

    del model
    del best_model
    del ema
    del opt
    del sch
    del scaler

    gc.collect()

    torch.cuda.empty_cache()


# ======================================================
# DISPLAY FOLD RESULTS
# ======================================================

fold_results = pd.DataFrame(
    fold_rows
)

display(
    fold_results
)


FOLD 0
Best checkpoint:   /kaggle/working/convnext_base.fb_in22k_ft_in1k_384_fold0_full_best.pt
Resume checkpoint: /kaggle/working/convnext_base.fb_in22k_ft_in1k_384_fold0_full_resume.pt


model.safetensors: reconstructing file:   0%|          |  0.00B /  354MB            

model.safetensors: downloading bytes:           |  0.00B            

No resume checkpoint. Starting Fold 0 from epoch 1.


✓ New best model saved: 1.85633
Fold 0 | Epoch 1/14 | Train Loss: 1.0763 | Val LogLoss: 1.85633 | Val Acc: 0.2885 | Best: 1.85633 | Best Epoch: 1 | Stale: 0/4 | Time: 14.7 min
✓ Resume checkpoint saved: Epoch 1


...

Fold 4 | Epoch 11/14 | Train Loss: 0.4792 | Val LogLoss: 0.84837 | Val Acc: 0.7495 | Best: 0.84261 | Best Epoch: 9 | Stale: 2/4 | Time: 9.2 min
✓ Resume checkpoint saved: Epoch 11


Fold 4 | Epoch 12/14 | Train Loss: 0.4758 | Val LogLoss: 0.85181 | Val Acc: 0.7490 | Best: 0.84261 | Best Epoch: 9 | Stale: 3/4 | Time: 9.3 min
✓ Resume checkpoint saved: Epoch 12


Fold 4 | Epoch 13/14 | Train Loss: 0.4593 | Val LogLoss: 0.85431 | Val Acc: 0.7504 | Best: 0.84261 | Best Epoch: 9 | Stale: 4/4 | Time: 9.3 min
✓ Resume checkpoint saved: Epoch 13
Early stopping at epoch 13

Loading best Fold 4 model...
✓ Best Fold 4 model loaded.


Fold 4 TTA LogLoss: 0.83328
Fold 4 TTA Accuracy: 0.7460


,fold,best_epoch,best_val_logloss,tta_logloss,tta_accuracy
0,0,5,1.259821,1.249886,0.606798
1,1,10,0.749721,0.725753,0.805439
2,2,11,0.857223,0.838084,0.732337
3,3,8,0.859720,0.849652,0.716247
4,4,9,0.842606,0.833285,0.746006


# OOF temperature calibration

In [8]:
mask=~np.isnan(oof).any(1); y=y_all[mask]; z=oof[mask]
raw_ll,raw_acc=score(y,z)
res=minimize_scalar(lambda T:score(y,z,T)[0],bounds=(.5,5.),method='bounded')
T=float(res.x); cal_ll,_=score(y,z,T)
print('OOF rows:',mask.sum(),'/',len(train_df))
print('Raw OOF logloss:',raw_ll)
print('Best temperature:',T)
print('Calibrated OOF logloss:',cal_ll)
np.save('/kaggle/working/oof_logits_stage1.npy',oof)
np.save('/kaggle/working/oof_targets_stage1.npy',y_all)
np.save('/kaggle/working/test_logits_folds_stage1.npy',np.stack(test_fold_logits))
pd.DataFrame(fold_rows).to_csv('/kaggle/working/fold_scores_stage1.csv',index=False)

OOF rows: 16488 / 16488
Raw OOF logloss: 0.880932451081636
Best temperature: 1.0407886654863103
Calibrated OOF logloss: 0.8799178769188115


# Create raw and calibrated DrivenData submissions

In [9]:
raw=np.mean([softmax_np(z,1.) for z in test_fold_logits],axis=0)
cal=np.mean([softmax_np(z,T) for z in test_fold_logits],axis=0)

def make_sub(probs,path):
    p=pd.DataFrame(probs,columns=class_names); p.insert(0,'id',test_ids)
    s=submission_format[['id']].merge(p,on='id',validate='one_to_one')[submission_format.columns]
    assert np.allclose(s[class_names].sum(1),1.,atol=1e-5)
    s.to_csv(path,index=False); return s

make_sub(raw,'/kaggle/working/submission_stage1_raw.csv')
sub=make_sub(cal,'/kaggle/working/submission_stage1_calibrated.csv')
print('Saved submission_stage1_raw.csv and submission_stage1_calibrated.csv')
print('Temperature:',T)
display(sub.head())

Saved submission_stage1_raw.csv and submission_stage1_calibrated.csv
Temperature: 1.0407886654863103


,id,antelope_duiker,bird,blank,civet_genet,hog,leopard,monkey_prosimian,rodent
0,ZJ016488,0.050529,0.020999,0.611191,0.135957,0.032579,0.096523,0.017245,0.034976
1,ZJ016489,0.344850,0.047612,0.106086,0.088783,0.059921,0.080213,0.082521,0.190015
2,ZJ016490,0.156500,0.041163,0.073613,0.228737,0.330129,0.079541,0.037700,0.052616
3,ZJ016491,0.016155,0.012261,0.017694,0.017220,0.016688,0.891853,0.013272,0.014857
4,ZJ016492,0.032693,0.031466,0.021461,0.019698,0.037443,0.038115,0.127218,0.691905


In [10]:

for f in os.listdir("/kaggle/working"):
    if "fold0" in f and f.endswith(".pt"):
        print(f)

convnext_base.fb_in22k_ft_in1k_384_fold0_full_resume.pt
convnext_base.fb_in22k_ft_in1k_384_fold0_full_best.pt


In [11]:
checkpoint = torch.load(
    resume_ck,
    map_location="cpu",
    weights_only=False
)

print("Saved epoch:", checkpoint["epoch"])
print("Best epoch:", checkpoint["best_epoch"])
print("Best loss:", checkpoint["best"])
print("Run mode:", checkpoint["run_mode"])
print("Planned epochs:", checkpoint["planned_epochs"])

Saved epoch: 13
Best epoch: 9
Best loss: 0.8426064458502699
Run mode: full
Planned epochs: 14
